<a href="https://colab.research.google.com/github/Guiitens2005/GS_DynamicProgramming/blob/main/GSDyinamicProgramming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Global Solution - Dynamic Programming
# Integrantes:
# - Caio Berardo de Araújo, RM: 560357
# - Giovanni Romano Provazi, RM: 560434
# - Guilherme Augusto Caseiro, RM: 559765
# - Leonardo Fernandes Mesquita, RM: 559623
# - Vitor de Lima Domingues, RM: 561008

import heapq
import math
import time


# Classe do Sistema do Satélite
class Satellite:

    def __init__(self, name, x, y, battery):
        self.name = name
        self.x = x  # Coordenada X simulada no espaço
        self.y = y  # Coordenada Y simulada no espaço
        self.battery = battery  # Nível de bateria (0 a 100%)

    def get_status(self):
        return f"{self.name:15} | Posição: ({self.x:3}, {self.y:3}) | Bateria: {self.battery}%"


# Classe do Sistema de Logística Espacial
class SpaceLogisticsSystem:

    def __init__(self):
        self.constellation = []

    def add_satellite(self, satellite):
        self.constellation.append(satellite)
        print(f"\n[SUCESSO] {satellite.name} integrado à rede de monitoramento.")

    def list_satellites(self):
        if not self.constellation:
            print("\n[AVISO] Nenhum satélite cadastrado na rede ainda.")
            return
        print("\n--- SATÉLITES EM ÓRBITA ---")
        for sat in self.constellation:
            print(sat.get_status())
        print("-" * 50)

    def calculate_distance(self, x1, y1, x2, y2):
        # Fórmula da distância euclidiana para simular proximidade
        return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

    def _build_graph(self, station_name, station_x, station_y):
        """
        Método auxiliar para construir uma lista de adjacência representando o grafo da rede.
        Conecta a estação terrestre aos satélites e os satélites entre si.
        """
        graph = {}

        # O nó de partida é a Estação Terrestre
        graph[station_name] = []

        # 1. Conectar a Estação Terrestre a todos os satélites operacionais
        for sat in self.constellation:
            if sat.battery < 20:
                continue  # Ignora satélites com bateria baixa
            distance = self.calculate_distance(
                station_x, station_y, sat.x, sat.y
            )
            graph[station_name].append((distance, sat.name))

        # 2. Conectar os satélites entre si (Simulação de Rede Mesh)
        for sat1 in self.constellation:
            if sat1.battery < 20:
                continue
            if sat1.name not in graph:
                graph[sat1.name] = []

            for sat2 in self.constellation:
                if sat1 == sat2 or sat2.battery < 20:
                    continue
                distance = self.calculate_distance(
                    sat1.x, sat1.y, sat2.x, sat2.y
                )
                graph[sat1.name].append((distance, sat2.name))

        return graph

    def optimize_connection(self, station_name, station_x, station_y):
        """
        Encontra o caminho de rede mais curto da Estação Terrestre até o
        satélite alvo principal (FIAP-Sat Alpha) usando o Algoritmo de Dijkstra.
        """
        if not self.constellation:
            print(
                "\n[ERRO] Impossível otimizar. Nenhum satélite cadastrado no sistema."
            )
            return

        # Destino final para o caminho de transmissão de dados
        target_satellite = "FIAP-Sat Alpha"

        print(
            f"\n[DIJKSTRA] Buscando melhor rota de {station_name} até {target_satellite}..."
        )
        time.sleep(0.8)

        # Constrói o layout da rede baseado nas coordenadas atuais
        graph = self._build_graph(station_name, station_x, station_y)

        # Verificação de segurança: Se o satélite de destino estiver offline ou ausente
        if target_satellite not in graph:
            print(
                f"\n[ERRO] O satélite alvo ({target_satellite}) está inoperante ou com bateria baixa!"
            )
            return

        # --- IMPLEMENTAÇÃO DO ALGORITMO DE DIJKSTRA ---
        # A fila de prioridade armazena tuplas: (distancia_acumulada, no_atual)
        priority_queue = [(0, station_name)]

        distances = {node: float("inf") for node in graph}
        distances[station_name] = 0

        predecessors = {node: None for node in graph}

        while priority_queue:
            current_distance, current_node = heapq.heappop(priority_queue)

            # Para a busca se chegarmos com sucesso ao nosso destino
            if current_node == target_satellite:
                break

            # Ignora caminhos desatualizados na fila de prioridade
            if current_distance > distances.get(current_node, float("inf")):
                continue

            # Avalia as conexões com os nós vizinhos
            for edge_weight, neighbor in graph.get(current_node, []):
                new_distance = current_distance + edge_weight

                if (
                    neighbor in distances
                    and new_distance < distances[neighbor]
                ):
                    distances[neighbor] = new_distance
                    predecessors[neighbor] = current_node
                    heapq.heappush(
                        priority_queue, (new_distance, neighbor)
                    )

        # --- RECONSTRUÇÃO DO CAMINHO ---
        if distances.get(target_satellite) == float("inf"):
            print("\n" + "=" * 50)
            print(
                "   [ERRO] Não foi possível estabelecer uma rota de sinal até o satélite alvo!"
            )
            print("=" * 50)
            return

        path = []
        step = target_satellite
        while step is not None:
            path.append(step)
            step = predecessors.get(step)
        path.reverse()  # Inverte o array para ler do Início -> Fim

        # --- INTERFACE DE SAÍDA ---
        print("\n" + "=" * 50)
        print("   ROTA OTIMIZADA ENCONTRADA (DIJKSTRA)!")
        print(f"   Origem:   {station_name}")
        print(f"   Destino:  {target_satellite}")
        print(f"   Caminho:  {' -> '.join(path)}")
        print(
            f"   Distância Total do Sinal: {distances[target_satellite]:.2f} unidades"
        )
        print("=" * 50)


# FUNÇÃO DO MENU EM PORTUGUÊS
def show_menu():
    print("\n" + "#" * 40)
    print("      SPACE CONNECT - GLOBAL SOLUTION")
    print("#" * 40)
    print("[1] Cadastrar Novo Satélite")
    print("[2] Listar Constelação Atual")
    print("[3] Otimizar Conexão com Estação Terrestre")
    print("[0] Sair do Sistema")
    print("#" * 40)


# Loop Principal da Aplicação
if __name__ == "__main__":
    system = SpaceLogisticsSystem()

    # Dados simulados pré-carregados
    system.constellation.append(
        Satellite("FIAP-Sat Alpha", x=12, y=35, battery=85)
    )
    system.constellation.append(
        Satellite("FIAP-Sat Beta", x=45, y=50, battery=15)
    )
    system.constellation.append(
        Satellite("FIAP-Sat Gamma", x=18, y=22, battery=90)
    )

    while True:
        show_menu()  # O menu é chamado aqui a cada ciclo do loop
        option = input("Escolha uma opção (0-3): ").strip()

        # [1] Cadastrar Novo Satélite
        if option == "1":
            print("\n--- CADASTRO DE NOVO SATÉLITE ---")
            try:
                name = input("Nome do Satélite: ").strip()
                x = int(input("Coordenada X da órbita: "))
                y = int(input("Coordenada Y da órbita: "))
                battery = int(input("Porcentagem da Bateria (0-100): "))

                if 0 <= battery <= 100:
                    new_satellite = Satellite(name, x, y, battery)
                    system.add_satellite(new_satellite)
                else:
                    print("\n[ERRO] A bateria deve ser um valor entre 0 e 100.")
            except ValueError:
                print(
                    "\n[ERRO] Entrada inválida! Coordenadas e bateria devem ser números inteiros."
                )

        # [2] Listar Constelação Atual
        elif option == "2":
            system.list_satellites()

        # [3] Otimizar Conexão via Dijkstra
        elif option == "3":
            print("\n--- LOCALIZAÇÃO DA ESTAÇÃO TERRESTRE ---")
            try:
                station_name = input(
                    "Nome da Estação (ex: Base Amazônia): "
                ).strip()
                station_x = int(input("Digite a coordenada X da estação: "))
                station_y = int(input("Digite a coordenada Y da estação: "))

                system.optimize_connection(station_name, station_x, station_y)
            except ValueError:
                print(
                    "\n[ERRO] Entrada inválida! As coordenadas devem ser números inteiros."
                )

        # [0] Sair do Sistema
        elif option == "0":
            print("\nDesconectando do Space Connect... Sistema encerrado.")
            break

        else:
            print(
                "\n[AVISO] Opção inválida! Por favor, escolha um número de 0 a 3."
            )

        input("\nPressione ENTER para continuar...")


########################################
      SPACE CONNECT - GLOBAL SOLUTION
########################################
[1] Cadastrar Novo Satélite
[2] Listar Constelação Atual
[3] Otimizar Conexão com Estação Terrestre
[0] Sair do Sistema
########################################
Escolha uma opção (0-3): 3

--- LOCALIZAÇÃO DA ESTAÇÃO TERRESTRE ---
Nome da Estação (ex: Base Amazônia): 3
Digite a coordenada X da estação: 3
Digite a coordenada Y da estação: 1

[DIJKSTRA] Buscando melhor rota de 3 até FIAP-Sat Alpha...

   ROTA OTIMIZADA ENCONTRADA (DIJKSTRA)!
   Origem:   3
   Destino:  FIAP-Sat Alpha
   Caminho:  3 -> FIAP-Sat Alpha
   Distância Total do Sinal: 35.17 unidades

Pressione ENTER para continuar...

########################################
      SPACE CONNECT - GLOBAL SOLUTION
########################################
[1] Cadastrar Novo Satélite
[2] Listar Constelação Atual
[3] Otimizar Conexão com Estação Terrestre
[0] Sair do Sistema
########################################